# LAB-D1-03: Shape and Activation Observatory

**Purpose:** Make dense-layer shapes, parameter counts, bias broadcasting, activation behavior, and class-axis normalization observable before assembling a complete network.

**Objectives:** `OBJ-D1-02`, `OBJ-D1-05`, `OBJ-D1-06`  
**Estimated duration:** 40 minutes live; under 10 seconds compute  
**Prerequisites:** `LESSON-D1-04`, `LESSON-D1-05`, `ACT-D1-03`, `ACT-D1-04`; batch-first matrix notation  
**Environment:** CPU only; NumPy and matplotlib; generated arrays; no network or download

Workflow: **Observe -> Predict -> Modify -> Run -> Visualize -> Diagnose -> Explain -> Extend**. Restart and run in order. The notebook is standalone.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 31
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__}")
print("Runtime target: local/Colab CPU; no network or GPU required.")

## Observe: Architecture and Starter State

The architecture is `4 -> 8 -> 3`. Five examples occupy the batch dimension. Under the shared convention, examples are rows and a dense operation is `X @ W + b`.

The supplied matrices are seeded support data, not trained parameters. Your task is to implement and audit the mechanics.

In [ ]:
X = np.array([
    [0.5, -1.0, 0.3, 1.2],
    [-0.8, 0.2, 1.1, -0.4],
    [1.5, 0.7, -0.6, 0.1],
    [0.0, -1.3, 0.8, 0.9],
    [-1.1, 1.4, 0.2, -0.7],
])
W1 = rng.normal(0.0, 0.55, size=(4, 8))
b1 = rng.normal(0.0, 0.15, size=(8,))
W2 = rng.normal(0.0, 0.45, size=(8, 3))
b2 = rng.normal(0.0, 0.10, size=(3,))

print("Supplied object shapes:", {"X": X.shape, "W1": W1.shape, "b1": b1.shape, "W2": W2.shape, "b2": b2.shape})

## Predict: Shapes, Broadcasting, and Parameter Count

Before multiplying arrays, commit to the shapes of `Z1`, `A1`, `logits`, and `probabilities`. Calculate the total trainable parameters and explain whether batch size contributes. Predict what one bias vector does across five examples.

In [ ]:
shape_predictions = {
    "Z1": "",
    "A1": "",
    "logits": "",
    "probabilities": "",
    "parameter_count": "",
    "batch_and_broadcasting": "",
}
assert all(value.strip() for value in shape_predictions.values()), (
    "Prediction checkpoint: complete all shape/count/broadcasting fields before running dense operations."
)

## Modify: Parameter Count and Dense Broadcasting

Complete the two TODOs. `dense_parameter_count` should count weights plus biases for every adjacent width pair. `dense` should use one batch-first matrix multiplication and one broadcasted bias addition.

In [ ]:
def dense_parameter_count(layer_widths):
    # TODO: sum d_in*d_out + d_out for each adjacent pair.
    raise NotImplementedError("TODO: calculate the dense-network parameter count")


def dense(inputs, weights, bias):
    # TODO: return the batch-first affine transformation.
    raise NotImplementedError("TODO: implement inputs @ weights + bias")

In [ ]:
parameter_count = dense_parameter_count([4, 8, 3])
Z1 = dense(X, W1, b1)
Z1_without_bias = X @ W1
broadcast_delta = Z1 - Z1_without_bias

assert parameter_count == 67, "Recheck weights and biases for both layers."
assert Z1.shape == (5, 8)
assert np.allclose(broadcast_delta, np.broadcast_to(b1, Z1.shape))
print("Parameter count:", parameter_count)
print("Z1 shape:", Z1.shape)
print("First two broadcast deltas match the same bias vector:")
print(np.round(broadcast_delta[:2], 3))

## Predict: Activation Shapes and Extreme Behavior

For the shared input values from `-12` to `12`, predict the output range and the behavior at both extremes for sigmoid, tanh, and ReLU. Predict what a strongly negative matrix will produce after ReLU. State which operations preserve shape.

In [ ]:
activation_predictions = {
    "sigmoid_range_and_extremes": "",
    "tanh_range_and_extremes": "",
    "relu_negative_and_positive": "",
    "strongly_negative_matrix": "",
    "shape_invariant": "",
}
assert all(value.strip() for value in activation_predictions.values()), (
    "Prediction checkpoint: complete every activation field before plotting the curves."
)

## Modify: Implement Stable Activations

Complete each activation TODO for sigmoid, tanh, and ReLU. Preserve input shape. Sigmoid must remain finite for very large positive and negative values.

In [ ]:
def sigmoid(values):
    # TODO: implement a numerically stable elementwise sigmoid.
    raise NotImplementedError("TODO: implement stable sigmoid")


def tanh_activation(values):
    # TODO: implement elementwise tanh.
    raise NotImplementedError("TODO: implement tanh")


def relu(values):
    # TODO: implement elementwise ReLU.
    raise NotImplementedError("TODO: implement ReLU")

In [ ]:
activation_inputs = np.linspace(-12.0, 12.0, 401)
activation_functions = [
    ("sigmoid", sigmoid),
    ("tanh", tanh_activation),
    ("ReLU", relu),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True)
for ax, (name, function) in zip(axes, activation_functions):
    outputs = function(activation_inputs)
    assert outputs.shape == activation_inputs.shape and np.all(np.isfinite(outputs))
    ax.plot(activation_inputs, outputs, color="#1f5a7a", linewidth=2, label=name)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.axvline(0.0, color="black", linewidth=0.8)
    ax.set(title=name, xlabel="pre-activation z", ylabel="activation output")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
inspection_values = np.array([-12.0, -3.0, 0.0, 3.0, 12.0])
print(f"{'input':>8} {'sigmoid':>10} {'tanh':>10} {'ReLU':>10}")
for index, value in enumerate(inspection_values):
    print(f"{value:>8.1f} {sigmoid(inspection_values)[index]:>10.4f} {tanh_activation(inspection_values)[index]:>10.4f} {relu(inspection_values)[index]:>10.4f}")

extreme_sigmoid = sigmoid(np.array([-1000.0, 0.0, 1000.0]))
assert np.all(np.isfinite(extreme_sigmoid))
assert np.all((0.0 <= extreme_sigmoid) & (extreme_sigmoid <= 1.0))

## Run and Diagnose: Dead Current Region and Saturation

Before running, distinguish these claims: `all outputs for this supplied negative matrix will be zero` and `this unit is permanently dead for every possible input`. Predict which claim the evidence can support. Also predict where sigmoid and tanh output changes will become visually small.

In [ ]:
strongly_negative = -np.arange(1.0, 41.0).reshape(5, 8)
negative_relu = relu(strongly_negative)
sigmoid_tail_change = float(np.diff(sigmoid(np.array([8.0, 12.0])))[0])
tanh_tail_change = float(np.diff(tanh_activation(np.array([8.0, 12.0])))[0])
assert negative_relu.shape == (5, 8)
assert np.count_nonzero(negative_relu) == 0
assert sigmoid_tail_change < 1e-3 and tanh_tail_change < 1e-3
print(f"Dead-region evidence: ReLU nonzero outputs = {np.count_nonzero(negative_relu)}/{negative_relu.size}")
print(f"Saturation evidence: sigmoid(8)->sigmoid(12) change = {sigmoid_tail_change:.6f}")
print(f"Saturation evidence: tanh(8)->tanh(12) change = {tanh_tail_change:.9f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
images = [
    (strongly_negative, "pre-activation: strongly negative"),
    (negative_relu, "ReLU output: current all-zero region"),
    (sigmoid(strongly_negative), "sigmoid output: compressed near zero"),
]
for ax, (values, title) in zip(axes, images):
    image = ax.imshow(values, aspect="auto", cmap="cividis")
    ax.set(title=title, xlabel="unit", ylabel="example")
    fig.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Predict: Multiclass Softmax and Its Axis

The output has three class logits per example. Predict the shape and the sum that should equal one for each example. Then predict what a softmax normalized down axis `0` would sum instead. Explain why both versions can return shape `(5, 3)` even though only one matches the task semantics.

In [ ]:
softmax_predictions = {
    "logit_and_probability_shape": "",
    "correct_axis_sums": "",
    "wrong_axis_sums": "",
    "why_shape_is_insufficient": "",
}
assert all(value.strip() for value in softmax_predictions.values()), (
    "Prediction checkpoint: complete all softmax fields before normalization."
)

## Modify: Implement Stable Row-Wise Softmax

Complete `softmax`. Normalize classes within each example (`axis=1`) and subtract a per-row maximum before exponentiating. Preserve `(batch, classes)` shape.

In [ ]:
def softmax(logits):
    # TODO: implement stable per-example, row-wise softmax.
    raise NotImplementedError("TODO: implement stable row-wise softmax")

In [ ]:
A1 = relu(Z1)
logits = dense(A1, W2, b2)
probabilities = softmax(logits)

assert A1.shape == (5, 8)
assert logits.shape == (5, 3)
assert probabilities.shape == (5, 3)
assert np.all(np.isfinite(probabilities))
assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
print("Z1/A1/logits/probabilities:", Z1.shape, A1.shape, logits.shape, probabilities.shape)
print("Softmax row sums:", np.round(probabilities.sum(axis=1), 8))

## Break It: Wrong-Axis Softmax

The supplied function below is deliberately wrong for this task: it normalizes across examples. It should run and preserve shape. The diagnostic assertion is expected to fail, and the cell catches that expected failure so you can inspect its message before recovering with your row-wise implementation.

In [ ]:
def wrong_axis_softmax(logits):
    shifted = logits - np.max(logits, axis=0, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=0, keepdims=True)


wrong_probabilities = wrong_axis_softmax(logits)
assert wrong_probabilities.shape == probabilities.shape
expected_failure_observed = False
try:
    assert np.allclose(wrong_probabilities.sum(axis=1), 1.0, atol=1e-6), (
        "Per-example row sums are not one; check the softmax normalization axis."
    )
except AssertionError as error:
    expected_failure_observed = True
    print("Expected deliberate failure:", error)

print("Wrong-axis row sums:", np.round(wrong_probabilities.sum(axis=1), 6))
print("Wrong-axis column sums:", np.round(wrong_probabilities.sum(axis=0), 6))
assert expected_failure_observed, "The deliberate wrong-axis case should violate row normalization."

In [ ]:
extreme_logits = np.array([[1000.0, 1001.0, 999.0], [-1000.0, -1001.0, -999.0]])
recovered_probabilities = softmax(extreme_logits)
assert recovered_probabilities.shape == (2, 3)
assert np.all(np.isfinite(recovered_probabilities))
assert np.allclose(recovered_probabilities.sum(axis=1), 1.0, atol=1e-6)
print("Recovery row sums under extreme logits:", recovered_probabilities.sum(axis=1))

## Diagnose and Explain

Use observed arrays or plots to answer:

1. Why is the parameter count independent of five examples?
2. What evidence shows one bias vector broadcasting across rows?
3. Where did sigmoid/tanh compress changes, and what did ReLU do to the supplied negative region?
4. Why does the all-zero ReLU result describe the current input region rather than every possible input?
5. Which invariant caught the softmax defect that shape checks missed?
6. How do hidden-layer and multiclass-output roles differ?

In [ ]:
explanation = {
    "batch_vs_parameters": "",
    "broadcast_evidence": "",
    "activation_evidence": "",
    "current_vs_permanent_relu": "",
    "softmax_invariant": "",
    "hidden_vs_output_role": "",
}
assert all(value.strip() for value in explanation.values()), "Explanation checkpoint: complete all six evidence statements."
assert parameter_count == 67
assert Z1.shape == A1.shape == (5, 8)
assert logits.shape == probabilities.shape == (5, 3)
assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
assert expected_failure_observed and np.count_nonzero(negative_relu) == 0
print("LAB-D1-03 checkpoint passed: 67 parameters, dense broadcasting, activation evidence, and softmax failure/recovery.")

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Parameter count is not 67 | A bias vector was skipped or batch size was counted | Count `d_in*d_out + d_out` for both dense layers |
| Dense matrix error | Examples or weights use the wrong orientation | Restore `(B,d_in) @ (d_in,d_out)` |
| Bias result has an extra dimension | Bias was reshaped inconsistently | Use the supplied `(d_out,)` vector and NumPy row broadcasting |
| Sigmoid overflows | Direct exponentiation used an unstable sign | Branch by sign or use an equivalent stable expression |
| Wrong-axis cell prints a failure | This is the intended diagnostic | Inspect row/column sums, then verify recovery with `softmax` |
| Softmax produces NaN or infinity | Per-row maximum was not subtracted | Shift each row before exponentiating |

## Takeaways

- The `4 -> 8 -> 3` architecture contains 67 trainable parameters, independent of batch size.
- Dense bias broadcasting reuses one parameter vector across examples.
- Elementwise activations preserve shape but produce different ranges and failure evidence.
- Stable probability functions must remain finite under extreme inputs.
- Shape correctness is necessary but not sufficient; semantic invariants expose wrong axes.

Return to the [LAB-D1-03 debrief](../student-guide/day-1-student-guide.md#lab-d1-03---shape-and-activation-observatory). Review [LESSON-D1-04](../student-guide/day-1-student-guide.md#lesson-d1-04---build-a-network-from-layers), [LESSON-D1-05](../student-guide/day-1-student-guide.md#lesson-d1-05---activation-functions-and-nonlinearity), [ACT-D1-03](../challenges/day-1-challenges.md#act-d1-03---activation-card-sort), and [ACT-D1-04](../challenges/day-1-challenges.md#act-d1-04---shape-relay) as needed.

---

## OPTIONAL EXTENSION: Leaky ReLU

**The core lab is complete.** Everything below is independently skippable. Skipping every optional cell does not affect any core checkpoint or later core work.

Implement Leaky ReLU, compare it with ReLU on the same values, and change only its negative slope. Before running the evidence cell, predict:

1. Which outputs differ from ReLU when `negative_slope=0.1`?
2. Which values and shapes remain invariant when the slope changes from `0.1` to `0.02`?
3. What visible evidence would show that the negative side retains a nonzero slope?

**Hint:** `np.where` can choose one elementwise expression for nonnegative values and another for negative values.

In [ ]:
optional_leaky_relu_predictions = {
    "relu_comparison": "",
    "slope_change_invariants": "",
    "evidence_of_negative_slope": "",
}
assert all(value.strip() for value in optional_leaky_relu_predictions.values()), (
    "Optional prediction checkpoint: complete all three fields before revealing the comparison."
)

In [ ]:
def leaky_relu(values, negative_slope=0.1):
    # OPTIONAL TODO: retain the supplied small slope for negative values.
    raise NotImplementedError("OPTIONAL TODO: implement Leaky ReLU")


leaky_default = leaky_relu(activation_inputs, negative_slope=0.1)
leaky_shallow = leaky_relu(activation_inputs, negative_slope=0.02)
relu_outputs = relu(activation_inputs)

assert leaky_default.shape == leaky_shallow.shape == relu_outputs.shape == activation_inputs.shape
assert np.all(np.isfinite(leaky_default)) and np.all(np.isfinite(leaky_shallow))
assert np.allclose(leaky_default[activation_inputs >= 0], relu_outputs[activation_inputs >= 0])
assert np.allclose(leaky_shallow[activation_inputs >= 0], relu_outputs[activation_inputs >= 0])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(activation_inputs, relu_outputs, linewidth=2, label="ReLU")
ax.plot(activation_inputs, leaky_default, linewidth=2, label="Leaky ReLU, slope=0.10")
ax.plot(activation_inputs, leaky_shallow, linewidth=2, label="Leaky ReLU, slope=0.02")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.axvline(0.0, color="black", linewidth=0.8)
ax.set(title="Optional slope experiment", xlabel="pre-activation z", ylabel="activation output")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'input':>8} {'ReLU':>10} {'Leaky .10':>12} {'Leaky .02':>12}")
for index, value in enumerate(inspection_values):
    relu_value = relu(inspection_values)[index]
    default_value = leaky_relu(inspection_values, 0.1)[index]
    shallow_value = leaky_relu(inspection_values, 0.02)[index]
    print(f"{value:>8.1f} {relu_value:>10.4f} {default_value:>12.4f} {shallow_value:>12.4f}")

### Optional Interpretation

Use the plot and table to explain how Leaky ReLU differs from ReLU for negative inputs, how changing only the negative slope changes the evidence, and why this comparison does not establish that one activation is universally better.

In [ ]:
optional_leaky_relu_interpretation = {
    "negative_input_comparison": "",
    "slope_experiment_evidence": "",
    "limits_of_the_comparison": "",
}
assert all(value.strip() for value in optional_leaky_relu_interpretation.values()), (
    "Optional interpretation checkpoint: cite evidence from the plot or table in all three fields."
)
print("Optional Leaky ReLU extension checkpoint passed.")